# 03: CBM Training (Stage 2)

Train CBM-LR (sparse L1 logistic regression) and CBM-MLP on the Derm7pt concept-supervised subset, 5-fold stratified CV.

In [1]:
import os, sys, yaml, json, logging
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np, pandas as pd
from src.models.concept_predictor import load_concept_bundle, CONCEPT_IDS
from src.models.cbm_classifier import CBMLogisticRegression, CBMMLP, stratified_cv
from src.utils import seed_everything

logging.basicConfig(level=logging.INFO)
cfg = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
paths = cfg['paths']
seed_everything(cfg.get('seed', 42))

bundle = load_concept_bundle(PROJECT_ROOT / paths['concept_vectors_dir'] / 'concepts_derm7pt.npz')
X, y = bundle['concepts'], bundle['diagnosis']
print('X:', X.shape, ' positive rate:', y.mean())

X: (1011, 9)  positive rate: 0.29080118694362017


In [2]:
# 5-fold CV for CBM-LR, grid over C
def make_lr():
    return CBMLogisticRegression(penalty=cfg['cbm_lr']['penalty'],
                                 C=cfg['cbm_lr']['C'],
                                 class_weight=cfg['cbm_lr']['class_weight'])
cv_lr = stratified_cv(make_lr, X, y, n_folds=cfg['cbm_lr']['cv_folds'])
print('CBM-LR CV mean:', cv_lr.mean)
print('CBM-LR CV std: ', cv_lr.std)

# Refit on all data and tune C via GridSearchCV
lr = make_lr()
lr_cv = lr.cross_validate(X, y, C_grid=cfg['cbm_lr']['C_grid'])
print('best C:', lr_cv['best_C'], 'best score:', round(lr_cv['best_score'], 4))

CBM-LR CV mean: {'auroc': 0.7895015404914587, 'auprc': 0.5998738544874065, 'balanced_acc': 0.7235746906188169}
CBM-LR CV std:  {'auroc': 0.0288242840298016, 'auprc': 0.049249652153609554, 'balanced_acc': 0.020026664131199672}


/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave

best C: 0.1 best score: 0.7898


/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave

In [3]:
# Final CBM-LR trained on the full Derm7pt set
final_lr = make_lr()
final_lr.fit(X, y)
ckpt = PROJECT_ROOT / paths['checkpoints_dir'] / 'cbm_lr.joblib'
final_lr.save(ckpt); print('saved to', ckpt)
print('Concept weights:')
for k, v in final_lr.concept_weights().items():
    print(f'  {k:20s} {v:+.3f}')

saved to /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/model_checkpoints/cbm_lr.joblib
Concept weights:
  pigment_network      -0.589
  dots_globules        +0.007
  blue_white_veil      +0.673
  streaks              -0.002
  regression           -0.179
  vascular             +0.467
  asymmetry            +0.088
  border               +0.169
  color_var            +0.732


/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [4]:
# CBM-MLP (ablation)
def make_mlp():
    return CBMMLP(n_concepts=X.shape[1],
                  hidden_dim=cfg['cbm_mlp']['hidden_dim'],
                  dropout=cfg['cbm_mlp']['dropout'],
                  learning_rate=cfg['cbm_mlp']['learning_rate'],
                  weight_decay=cfg['cbm_mlp']['weight_decay'],
                  epochs=cfg['cbm_mlp']['epochs'],
                  batch_size=cfg['cbm_mlp']['batch_size'],
                  early_stopping_patience=cfg['cbm_mlp']['early_stopping_patience'])
cv_mlp = stratified_cv(make_mlp, X, y, n_folds=cfg['cbm_lr']['cv_folds'])
print('CBM-MLP CV mean:', cv_mlp.mean)
print('CBM-MLP CV std: ', cv_mlp.std)

final_mlp = make_mlp().fit(X, y)
mlp_ckpt = PROJECT_ROOT / paths['checkpoints_dir'] / 'cbm_mlp.pt'
final_mlp.save(mlp_ckpt); print('saved to', mlp_ckpt)

CBM-MLP CV mean: {'auroc': 0.7917218480266406, 'auprc': 0.6169947214730023, 'balanced_acc': 0.7163764777660161}
CBM-MLP CV std:  {'auroc': 0.018581452019860507, 'auprc': 0.022843814591320833, 'balanced_acc': 0.01369938278529025}
saved to /Users/deepesh/Documents/Claude/Projects/AIH Final/vlm-cbm-derm-fairness/outputs/model_checkpoints/cbm_mlp.pt


In [5]:
# Export CV summary
summary = {
    'cbm_lr': {'mean': cv_lr.mean, 'std': cv_lr.std, 'best_C': final_lr.C},
    'cbm_mlp': {'mean': cv_mlp.mean, 'std': cv_mlp.std},
}
out = PROJECT_ROOT / paths['tables_dir'] / 'cbm_cv_summary.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

{
  "cbm_lr": {
    "mean": {
      "auroc": 0.7895015404914587,
      "auprc": 0.5998738544874065,
      "balanced_acc": 0.7235746906188169
    },
    "std": {
      "auroc": 0.0288242840298016,
      "auprc": 0.049249652153609554,
      "balanced_acc": 0.020026664131199672
    },
    "best_C": 1.0
  },
  "cbm_mlp": {
    "mean": {
      "auroc": 0.7917218480266406,
      "auprc": 0.6169947214730023,
      "balanced_acc": 0.7163764777660161
    },
    "std": {
      "auroc": 0.018581452019860507,
      "auprc": 0.022843814591320833,
      "balanced_acc": 0.01369938278529025
    }
  }
}
